# Check 1: Genie Precision, Recall, and the Missing Threshold

This notebook runs **Check 1** from the Genie Test Protocol end to end.

The check asks Genie a plain inbound-transfer ranking question — the same question an analyst would ask when starting a fraud investigation — and then measures how accurate that answer is against the known fraud ground truth.

**Four phases:**

- **Phase 1** — Call Genie with the Check 1 question and display the result as a table
- **Phase 2** — Confirm the `account_labels` ground truth table is accessible
- **Phase 3** — Join the result against ground truth and compute precision and recall
- **Phase 4** — Explain why threshold sensitivity cannot be measured for this result

**Before running:** Set your `SPACE_ID` in the configuration cell below. The Genie Space must be connected to the tables created by `setup/upload_and_create_tables.sh`.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set SPACE_ID to your Genie Space ID before running.
# Find it in Databricks under Genie → your space → the ID in the URL.

SPACE_ID = "YOUR-GENIE-SPACE-ID"   # <-- replace this

CATALOG  = "graph-enriched-lakehouse"
SCHEMA   = "graph-enriched-schema"

In [ ]:
from datetime import timedelta

from databricks.sdk import WorkspaceClient
import pandas as pd

w = WorkspaceClient()
print("Connected to:", w.config.host)

In [ ]:
def ask_genie(question: str, conversation_id: str = None, timeout_seconds: int = 120) -> dict:
    """Submit a question to the configured Genie Space and return a dict with the result.

    Returns a dict with keys:
      conversation_id  – reuse this for follow-up questions
      message_id       – ID of this message
      status           – Genie response status
      sql              – the SQL Genie generated, or None
      df               – pandas DataFrame of results, or None
      text             – text response from Genie, or None
    """
    if conversation_id:
        message = w.genie.create_message_and_wait(
            space_id=SPACE_ID,
            conversation_id=conversation_id,
            content=question,
            timeout=timedelta(seconds=timeout_seconds),
        )
    else:
        message = w.genie.start_conversation_and_wait(
            space_id=SPACE_ID,
            content=question,
            timeout=timedelta(seconds=timeout_seconds),
        )

    result = {
        "conversation_id": message.conversation_id,
        "message_id": message.id,
        "status": str(message.status.value) if message.status else "UNKNOWN",
        "sql": None,
        "df": None,
        "text": None,
    }

    if not message.attachments:
        return result

    for attachment in message.attachments:
        if attachment.text:
            result["text"] = attachment.text.content

        if attachment.query and attachment.attachment_id:
            result["sql"] = attachment.query.query
            data_result = w.genie.get_message_query_result_by_attachment(
                space_id=SPACE_ID,
                conversation_id=message.conversation_id,
                message_id=message.id,
                attachment_id=attachment.attachment_id,
            )
            sr = data_result.statement_response
            if sr and sr.manifest and sr.result:
                columns = [c.name for c in sr.manifest.schema.columns]
                rows = sr.result.data_array or []
                result["df"] = pd.DataFrame(rows, columns=columns)

    return result

## Phase 1 — Call Genie and Display Results

The question below gives Genie explicit fraud framing — asking about hubs of a money movement network that are potentially fraudulent. This is a stronger prompt than a plain inbound-count ranking, and Genie will likely surface some genuine fraud ring members alongside whale accounts.

The expected result is a mixed list: some fraud ring members from different rings, some whale accounts. The result looks more credible than a pure whale list — which makes the precision and recall analysis in Phase 3 more important, not less.

In [ ]:
CHECK_1_QUESTION = (
    "Are there accounts that seem to be the hub of a "
    "money movement network that are potentially fraudulent?"
)

print("Question:", CHECK_1_QUESTION)
print()

response = ask_genie(CHECK_1_QUESTION)
print(f"Status:  {response['status']}")

if response["text"] and response["df"] is None:
    print(f"\nGenie returned a text response instead of a table:\n{response['text']}")
    raise RuntimeError(
        "Genie did not return a query result. "
        "Confirm the Genie Space is connected to the correct tables and try again."
    )

print(f"Rows returned: {len(response['df'])}")

In [ ]:
# Genie's answer — the raw result before any comparison
print("Genie returned these accounts as the top inbound-transfer recipients:\n")
display(response["df"])

In [ ]:
# The SQL Genie generated — shows what Genie actually measured
print("SQL Genie generated:\n")
print(response["sql"])

## Phase 2 — Confirm the Ground Truth Table

The `account_labels` table holds a fraud label for every account in the dataset — one row per account with `account_id` and `is_fraud`. It was loaded by `setup/upload_and_create_tables.sh` from `data/account_labels.csv`.

This table provides the denominator for recall: the total number of accounts labeled as fraud, which is the population Genie's answer is measured against.

In [ ]:
labels_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.account_labels").toPandas()
print(f"account_labels: {len(labels_df):,} rows\n")
display(labels_df.head(5))

In [ ]:
total_accounts = len(labels_df)
total_fraud    = int(labels_df["is_fraud"].sum())

print(f"Total accounts in dataset:  {total_accounts:,}")
print(f"Total labeled as fraud:     {total_fraud:,}")
print(f"Fraud rate:                 {total_fraud / total_accounts:.1%}")
print()
print(f"{total_fraud:,} is the recall denominator — the number of accounts")
print("Genie would need to return to achieve 100% recall.")

## Phase 3 — Compute Precision and Recall

Genie returned a list of account IDs. Joining that list against the ground truth labels answers two questions:

- **Precision:** of the accounts Genie flagged, what fraction are actually fraud?
- **Recall:** of all known fraud accounts, what fraction did Genie return?

Precision may be moderate — the explicit fraud framing means Genie will find some real ring members. Recall will still be near zero: even if Genie returns fraud accounts, they represent a tiny fraction of the 1,000 ring members in the dataset. The deeper problem, shown in Phase 4, is that neither number can be improved by adjusting a threshold.

In [ ]:
# Identify the account ID column in Genie's result.
# Genie may name it 'account_id', 'dst_account_id', or similar depending on the SQL it wrote.
id_col = next(
    (c for c in response["df"].columns if "account_id" in c.lower()),
    response["df"].columns[0],
)
print(f"Using column '{id_col}' from Genie's result as the account identifier\n")

genie_ids = (
    response["df"][[id_col]]
    .rename(columns={id_col: "account_id"})
    .assign(account_id=lambda df: df["account_id"].astype("int64"))
)

genie_with_labels = genie_ids.merge(labels_df, on="account_id", how="left")

print("Genie's result with fraud labels:\n")
display(genie_with_labels)

In [ ]:
true_positives  = int(genie_with_labels["is_fraud"].sum())
total_returned  = len(genie_with_labels)
precision       = true_positives / total_returned if total_returned > 0 else 0.0

print(f"Precision: {precision:.1%}")
print(f"  {true_positives} fraud account(s) in Genie's top {total_returned} results")
print()
print("What this means: of the accounts Genie identified as high-inbound-transfer,")
print(f"{precision:.1%} are actually fraud ring members. The rest are whale accounts —")
print("high-volume legitimate accounts that dominate raw inbound count because they")
print("receive transfers from many peripheral senders.")

In [ ]:
recall = true_positives / total_fraud if total_fraud > 0 else 0.0

print(f"Recall: {recall:.1%}")
print(f"  {true_positives} of {total_fraud:,} known fraud accounts returned by Genie")
print()
print("What this means: Genie's answer covered only a tiny fraction of the")
print(f"{total_fraud:,} fraud ring members in the dataset. The remaining ring members")
print("are invisible to this query because raw inbound count cannot separate their")
print("transfer pattern from that of normal high-volume accounts.")

## Phase 4 — Why Threshold Cannot Be Measured

The precision and recall numbers above tell part of the story. What matters equally — and what this section makes explicit — is that there is **no way to improve those numbers by adjusting a threshold**.

### The structural problem

Genie returned a list of accounts with no score attached to any of them. Whatever signal Genie used internally to identify these accounts as suspicious hubs — transfer patterns, network structure, or some combination — that signal was not exposed as a column. Each account simply appears in the list or does not. There is no value to cut on.

In a standard classification system, a threshold is a number you draw across a score distribution. Accounts above the threshold are flagged; accounts below are not. You can slide the threshold up or down and observe the tradeoff: higher threshold means fewer false positives (better precision) at the cost of more false negatives (worse recall). The F1 score summarizes where the tradeoff is best. This is threshold sensitivity analysis.

**None of that applies here.** There is no score to cut. There is no distribution to draw a line across. The only parameter an analyst can change is the question itself — but re-asking a different question is not threshold analysis. There is no feedback signal to tell the analyst whether a rephrased question is better or worse without running through ground truth every time.

### What an analyst actually sees

The table in Phase 1 looks like a valid answer. It is a list of account IDs Genie believes are suspicious hubs. Without external ground truth, an analyst has no way to know how accurate it is — which accounts are genuine ring members, which are high-volume legitimate accounts, and which lie somewhere in between. The result is not obviously broken. It is unmeasurably inaccurate.

The moderate precision number from Phase 3 makes this harder to see, not easier. A result that is 40–50% accurate looks like it is working. But the analyst cannot know it is 40–50% accurate without ground truth they would not have in production. And they have no lever to push it higher — no score, no cutoff, no optimization target.

### Compare to Check 4 (PageRank)

After GDS enrichment, each account has a continuous `risk_score` derived from PageRank. Ring members score high because they receive transfers from other high-scoring ring members — the centrality compounds recursively through the ring topology. Whale accounts score moderate because their senders are peripheral low-PageRank nodes.

That score is a real probability-like signal. An analyst can set a threshold of 0.7 or 0.8 and measure what happens to precision and recall at each cutoff. The tradeoff curve is visible and the analyst can pick an operating point based on their review capacity. Threshold sensitivity analysis becomes possible — and the separation between fraud and non-fraud accounts is visible without needing ground truth at all.

In [ ]:
print("=" * 62)
print("CHECK 1 SUMMARY")
print("=" * 62)
print()
print(f"Genie returned {total_returned} accounts ranked by inbound transfer count.")
print()
print(f"  Precision: {precision:.1%}")
print(f"    {true_positives} of {total_returned} returned accounts are labeled fraud")
print()
print(f"  Recall:    {recall:.1%}")
print(f"    {true_positives} of {total_fraud:,} known fraud accounts were returned")
print()
print("-" * 62)
print("Threshold sensitivity: not measurable")
print("-" * 62)
print()
print("Genie's result carries no score. Every returned account has an")
print("inbound transfer count — an absolute integer — but no value that")
print("represents the probability of fraud. Without a score:")
print()
print("  - There is no cutoff to vary.")
print("  - There is no tradeoff curve to plot.")
print("  - There is no F1 score to optimize.")
print("  - An analyst cannot tell from the output alone which accounts")
print("    are whales and which are ring members.")
print()
print("The only lever available is changing N (top 20 vs top 50), which")
print("requires re-asking the question and does not improve precision —")
print("it only adds more whale accounts to a list that already has too many.")
print()
print("-" * 62)
print("Next: Check 4 (PageRank)")
print("-" * 62)
print()
print("After GDS enrichment the accounts table gains a continuous risk_score.")
print("That score enables the analysis this result cannot support: a threshold")
print("with a measurable precision/recall tradeoff, visible without ground truth.")